# 051 — Pretrained Models and ImageNet

Someone trained a very good CNN on 1.2 million photographs and gave the weights
away. This notebook checks the arithmetic of the most quoted of them: VGG16's
**138,357,544** parameters, **89.4%** of them dense and **74.3%** in one layer.

Computed from the architecture — no TensorFlow, no download.

| Part | What we check |
|---|---|
| A | the 13 convolutional layers |
| B | the three dense layers, and the total matching the published count |
| C | 89.4% dense, 74.3% in one layer |
| D | what training your own would cost instead |

## Part A — The convolutional stack

In [ ]:
BLOCKS = [(64, 2), (128, 2), (256, 3), (512, 3), (512, 3)]   # filters, repeats
INPUT = 224

channels, size, conv_params, n_conv = 3, INPUT, 0, 0
for filters, repeats in BLOCKS:
    for _ in range(repeats):
        conv_params += (3 * 3 * channels + 1) * filters
        channels = filters
        n_conv += 1
    size //= 2                       # one 2x2 max pool per block
print(f"{n_conv} convolutional layers: {conv_params:,} parameters")
print(f"final feature map: {size}x{size}x{channels}")
assert n_conv == 13 and conv_params == 14_714_688
assert (size, channels) == (7, 512)

## Part B — The classifier head, and the total

In [ ]:
flat = size * size * channels
dense = [(flat, 4096), (4096, 4096), (4096, 1000)]
dense_params = [i * o + o for i, o in dense]
total = conv_params + sum(dense_params)

print(f"flatten produces {flat:,} values")
for (i, o), p in zip(dense, dense_params):
    print(f"  Dense {i:>6} -> {o:<5} {p:>12,}  ({p / total:>5.1%})")
print(f"  {'TOTAL':<21} {total:>12,}")
assert flat == 25_088
assert total == 138_357_544      # VGG16's published count, exactly
print("matches the published count exactly - the arithmetic is right")

## Part C — The pattern, for the fourth time

In [ ]:
dense_share = sum(dense_params) / total
print(f"13 conv layers : {conv_params / total:.1%}")
print(f"3 dense layers : {dense_share:.1%}")
print(f"the first dense alone: {dense_params[0] / total:.1%}")
assert round(dense_share * 100, 1) == 89.4
assert round(dense_params[0] / total * 100, 1) == 74.3
print()
print("The layers that do the seeing are a tenth of the model. The layer facing")
print("the flattened 25,088-value vector is three quarters of it.")

## Part D — What you are being given

In [ ]:
# ILSVRC top-5 error, as reported by the competition - not measured here
winners = [(2010, 28.2, "shallow, hand-designed features"),
           (2011, 25.8, "shallow"),
           (2012, 16.4, "AlexNet - the discontinuity"),
           (2014, 7.3, "VGG16"),
           (2015, 3.6, "ResNet - below human error")]
for year, err, note in winners:
    print(f"  {year}  top-5 error {err:>4.1f}%   {note}")
drop = winners[1][1] - winners[2][1]
print(f"\n2011 to 2012: {drop:.1f} percentage points in one year")
assert round(drop, 1) == 9.4

print()
print("Using those weights costs you neither the 1.2 million labelled images")
print("nor the training time. Both are resources, not cleverness - which is the")
print("whole argument for transfer learning in lesson 053.")

## What this notebook established

- VGG16: 13 convolutional layers holding **14,714,688** parameters and three
  dense layers holding the rest, totalling **138,357,544** — the published
  figure, reproduced from the architecture alone.
- **89.4%** of the model is dense, and **74.3%** is the single layer facing the
  25,088-value flatten.
- ILSVRC top-5 error fell **9.4 points** in 2012 alone; these are reported
  competition results, not measured here.

## Exercises

1. Compute VGG19's total (blocks of 2, 2, 4, 4, 4). How much does the extra
   depth cost compared with the dense head?
2. VGG16 with `include_top=False` is 14,714,688 parameters. Check that against
   Part A and explain why it is exactly the conv total.
3. At what input size would the flatten produce 512 values, making the first
   dense layer cheap?